# Episodes: choosing θ and the gap tolerance g

`gatherings_v1` has one row per group per 5-minute slot, so a group that stays together for two hours appears as 24 rows. This notebook links such rows into **episodes** and chooses the two linking parameters.

**Definitions**

- Jaccard similarity of two member sets: $J(A,B)=|A\cap B|\,/\,|A\cup B|$.
- **θ**: two rows belong to the same group if $J \ge \theta$.
- **g**: a group may be unseen for at most $g$ consecutive slots and still continue.
- **Linking rule**: candidate links ($J\ge\theta$, at most $g$ skipped slots) are accepted shortest gap first, then highest $J$. Each row gets at most one successor and one predecessor. A chain of links is one episode.

**Measures** (there are no labelled meetings, so the evidence is internal)

| Measure | Meaning |
|---|---|
| `episodes` | number of episodes |
| `single_slot_%` | share of episodes seen in one slot only; high means strict linking |
| `median_slots`, `p95_slots`, `max_slots` | length of the episodes in slots |
| `bridged_gap_slots` | slots without a row inside episodes |
| `end_to_end_j` (long episodes) | $J$ between the first and last row; low means the group drifted |

An episode means continuous co-presence in the Bluetooth data, not interaction.

In [ ]:
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent                   # notebooks/ is one level below the project root
PROCESSED = PROJECT_ROOT / "data" / "processed"

sys.path.append(str(PROJECT_ROOT))                 # so "from src..." works inside notebooks/
from src.config import load_config
cfg = load_config(PROJECT_ROOT / "config" / "config.yaml")
SLOT_SEC = cfg["time_parameters"]["slot_duration_sec"]   # seconds per slot (300)

THETAS = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]            # similarity thresholds to try
GAPS = [0, 1, 2, 3]                                # numbers of skipped slots to tolerate

## 1. Linking functions

- `candidate_edges`: compares each row with the rows in the next $g+1$ slots and keeps the pairs with $J\ge\theta_{\min}$.
- `accept_links`: applies one $(\theta, g)$ setting and accepts the best links first.
- `link_rows`: follows the accepted links and returns the episodes.

In [ ]:
# the three linking functions now live in the project module
from src.features.episodes import candidate_edges, accept_links, link_rows

## 2. Check on a hand-linked example

The first 17 rows of `gatherings_v1` (slots 0 to 4) were linked by hand. The code must give 8 episodes at (θ 0.6, g 0), 6 at (0.6, 1), 10 at (0.8, 0) and 7 at (0.8, 1).

In [ ]:
toy_slots = [0,0,0, 1,1,1,1, 2,2,2, 3,3,3, 4,4,4,4]
toy_sets = [
    {12,244,454}, {176,221,263,472}, {215,311,524},
    {57,101,586}, {12,244,454}, {221,263,472}, {282,455,621},
    {12,244,454}, {176,221,263,472}, {282,455,621},
    {57,101,586}, {12,244,454}, {176,221,263,472},
    {64,90,91}, {57,101,586}, {176,221,263,452,472}, {282,455,621},
]
toy_edges = candidate_edges(toy_slots, toy_sets, 0.4, 3)
expected = {(0.6, 0): 8, (0.6, 1): 6, (0.8, 0): 10, (0.8, 1): 7}
for (theta, g), n_expected in expected.items():
    n = len(link_rows(toy_edges, len(toy_sets), theta, g))
    print(f"theta={theta} g={g}: {n} episodes (hand count {n_expected})")
    assert n == n_expected

## 3. Load the gatherings and compute candidate links once

In [ ]:
gath = pd.read_parquet(PROCESSED / "gatherings_v1.parquet").sort_values(["slot_id", "gathering_id"]).reset_index(drop=True)
slot_of = gath["slot_id"].to_numpy()                                         # slot of every row
members = [frozenset(int(u) for u in a) for a in gath["participant_set"]]    # member set of every row
print(f"{len(gath):,} slot-level gatherings, slots {slot_of.min()}..{slot_of.max()}")

t0 = time.time()
edges = candidate_edges(slot_of, members, min(THETAS), max(GAPS))            # computed once, loosest settings
print(f"{len(edges):,} candidate links in {time.time() - t0:.0f}s")

## 4. Grid over θ and g

One row per setting. `n_span` is the number of slots an episode covers, gaps included. Long episodes have `n_span` ≥ 5 (25 minutes or more).

In [ ]:
def episode_table(episodes):
    # one row per episode with its basic measures
    rows = []
    for chain in episodes:
        first_row = chain[0]
        last_row = chain[-1]
        first_slot = slot_of[first_row]
        last_slot = slot_of[last_row]
        n_span = last_slot - first_slot + 1                 # slots covered, gaps included

        everyone = set()                                    # all members that appear anywhere in the chain
        for k in chain:
            everyone = everyone | members[k]

        a = members[first_row]
        b = members[last_row]
        end_to_end_j = len(a & b) / len(a | b)              # how much the group drifted from start to end

        rows.append({"first_slot": first_slot, "last_slot": last_slot,
                     "n_rows": len(chain), "n_span": n_span,
                     "n_gap_slots": n_span - len(chain),    # slots covered but without a row
                     "union_size": len(everyone), "end_to_end_j": end_to_end_j})
    return pd.DataFrame(rows)


records = []
for g in GAPS:
    for theta in THETAS:
        episodes = link_rows(edges, len(gath), theta, g)
        ep = episode_table(episodes)
        long_ep = ep[ep.n_span >= 5]                        # episodes of 25 minutes or more
        records.append({
            "theta": theta, "gap": g, "episodes": len(ep),
            "single_slot_%": 100 * (ep.n_span == 1).mean(),
            "median_slots": ep.n_span.median(),
            "p95_slots": ep.n_span.quantile(.95),
            "max_slots": ep.n_span.max(),
            "bridged_gap_slots": int(ep.n_gap_slots.sum()),
            "end_to_end_j_median(long)": long_ep.end_to_end_j.median(),
            "end_to_end_j<0.5_%(long)": 100 * (long_ep.end_to_end_j < 0.5).mean(),
        })
grid = pd.DataFrame(records)
grid.round(2)

### Reading the grid

- Lower θ and a larger g give fewer, longer and more mixed episodes; higher θ and a smaller g give the opposite.
- A `max_slots` of 184 (about 15 hours) needs a check (section 7).
- A good setting is where `episodes` stops falling quickly while drift is still low.

In [ ]:
panels = {"episodes": "number of episodes",
          "single_slot_%": "episodes lasting one slot (%)",
          "end_to_end_j<0.5_%(long)": "long episodes with end-to-end J < 0.5 (%)"}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (column, label) in zip(axes, panels.items()):       # one panel per measure
    for g in GAPS:                                          # one line per gap
        sub = grid[grid.gap == g]
        ax.plot(sub.theta, sub[column], marker="o", label=f"gap={g}")
    ax.set(xlabel="theta", ylabel=label)
    ax.legend()
    ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

### Reading the plots

- **Episodes** rise with θ. A gap of 1 removes many of them; gaps 2 and 3 add little.
- **One-slot episodes** rise with θ, and a larger gap lowers them only slightly: at high θ many groups are never linked.
- **Drift** (long episodes with end-to-end $J<0.5$) is high at low θ, falls quickly between θ 0.5 and 0.7, and is about 0 from θ 0.8.
- **Trade-off:** a low θ drifts, a high θ fragments.

## 5. Are the bridged gaps silent phones?

For every bridged gap, we check whether each member seen on both sides logged any scan in the skipped slot (`bt_symmetric_prepared.parquet`), and compare with how often a phone is silent in a random slot. Members are active on both sides by construction, so this baseline is too high; section 6 corrects it.

In [ ]:
bt = pd.read_parquet(PROCESSED / "bt_symmetric_prepared.parquet", columns=["timestamp", "user_a"])
n_users = int(bt.user_a.max()) + 1
n_slots = int(slot_of.max()) + 1

# scanned[u, s] is True if phone u logged at least one row in slot s
scanned = np.zeros((n_users, n_slots), dtype=bool)
scan_slot = bt.timestamp.to_numpy() // SLOT_SEC             # slot of every logged row
scanned[bt.user_a.to_numpy(), scan_slot] = True

active_users = np.unique(bt.user_a)
baseline_silent = 100 * (1 - scanned[active_users].mean())  # share of (user, slot) cells with no scan
print(f"baseline: a participant's phone is silent in {baseline_silent:.1f}% of all slots")


def gap_silence(theta, g):
    # for every bridged link, check if the members seen on both sides logged anything in the skipped slots
    silent = 0
    total = 0
    successor = accept_links(edges, theta, g)
    for i, j in successor.items():
        skipped_slots = range(slot_of[i] + 1, slot_of[j])   # empty when the link skips no slot
        both_sides = members[i] & members[j]                # members present before and after the gap
        for u in both_sides:
            for s in skipped_slots:
                total += 1
                if not scanned[u, s]:
                    silent += 1
    return silent, total


for theta in (0.5, 0.6, 0.7):
    for g in (1, 2, 3):
        silent, total = gap_silence(theta, g)
        print(f"theta={theta} g={g}: {100 * silent / max(total, 1):.1f}% of member-slots inside bridged gaps were silent ({total:,} checked)")

## 6. A fairer baseline

The share of slots in which a phone is silent, given that it logged a scan in the slot before and the slot after. It applies to one-slot gaps only. A much higher silent share inside the gaps than here means that silent phones cause part of the gaps.

In [ ]:
sc = scanned[np.unique(bt.user_a)]                  # active phones only
before, middle, after = sc[:, :-2], sc[:, 1:-1], sc[:, 2:]
active_both_sides = before & after                  # phone logged in slot s-1 and s+1
silent_in_middle = 100 * (~middle[active_both_sides]).mean()
print(silent_in_middle)

## 7. Are the longest episodes overnight co-location or drift?

For the episodes of 3 hours or more at θ 0.7, g 1, we compute the start and end hour, the share between 00:00 and 07:00 and the end-to-end $J$. A high $J$ at night suggests a group sharing a place overnight; a low $J$ suggests drift. Hours assume that timestamp 0 is midnight, which the dataset does not state.

In [ ]:
episodes = link_rows(edges, len(gath), 0.7, 1)                 # chosen setting
ep = episode_table(episodes)
ep["start_hour"] = (ep.first_slot * SLOT_SEC // 3600) % 24     # assumes timestamp 0 = midnight
ep["end_hour"] = (ep.last_slot * SLOT_SEC // 3600) % 24

print(ep.sort_values("n_span", ascending=False).head(10)[["first_slot", "n_span", "union_size", "end_to_end_j", "start_hour", "end_hour"]])

long_ep = ep[ep.n_span >= 36]                                  # 3 hours or more
print(long_ep.start_hour.value_counts().sort_index())

def night_share(row):                                          # share of the episode's slots between 00:00 and 07:00
    slots = np.arange(row.first_slot, row.last_slot + 1)
    hours = (slots * SLOT_SEC // 3600) % 24
    return (hours < 7).mean()

long_ep = long_ep.assign(night=long_ep.apply(night_share, axis=1))
print((long_ep.night >= 0.8).mean(), long_ep.end_to_end_j.median())

### Findings: long episodes

- The ten longest episodes (up to 184 slots, about 15 hours) start between 18:00 and 23:00 and end between 07:00 and 11:00.
- They involve 3 to 6 people. The longest is the same three people throughout (end-to-end $J$ = 1.0).
- Episodes with an end-to-end $J$ of 0.2 to 0.6 are small groups in which one or two members come and go, not separate meetings chained together.
- Of the 392 episodes of 3 hours or more, 26% have at least 80% of their span between 00:00 and 07:00. Their median end-to-end $J$ is 0.77.

**Conclusion:** the long episodes are overnight co-location of small, stable groups. They are kept and flagged with `start_hour` and `night_share`. Bluetooth cannot show whether the place is a residence or the campus.

## 8. Should one-slot episodes be kept?

Most episodes (55%) consist of a single row. This block compares them with the other episodes. Result: they cover 12.6% of the rows, have a median of 4 students and a median RSSI of −82.5 dBm. They are kept, with `n_slots` as a column so that later steps can filter them.

In [ ]:
single = [chain[0] for chain in episodes if len(chain) == 1]   # rows that stayed alone
print(len(single) / len(gath))
sizes = gath.participant_set.apply(len)
print(sizes.iloc[single].describe(), gath.median_rssi.iloc[single].median())

## 9. Does the choice hold at RSSI −80 and −85?

Checked separately with `python -m src.rssi_sensitivity` at θ 0.7, g 1 (`outputs/tables/rssi_sensitivity_episodes.csv`):

| RSSI threshold | Episodes | One-slot share | Drift |
|---|---|---|---|
| −80 dBm | 23,028 | 61.0% | 4.5% |
| −85 dBm | 21,156 | 57.7% | 6.3% |
| −90 dBm | 22,530 | 54.9% | 6.3% |

The full grid over θ and g was not repeated at −80 and −85 dBm. The analysis uses −90 dBm.

## 10. Decision record

**Chosen: θ = 0.7, g = 1** (a group may be unseen for one slot, 5 minutes).

**Why θ = 0.7**
- Drift falls steeply up to 0.7 and then flattens: 17% at θ 0.6, 6% at 0.7, 1% at 0.8.
- Above 0.7 the episodes fragment: 29,411 at θ 0.8 against 22,530 at 0.7 (+31%), and one-slot episodes rise from 55% to 61%.
- In practice, a group of four can gain or lose one member and remain the same episode; swapping one member breaks the link.

**Why g = 1**
- Most of the gain comes from g 0 → 1 (−23% episodes); g 2 and 3 add only −11% and −6%.
- Larger gaps increase drift (6.3%, 7.1%, 7.6%) and the longest episode (from 184 to 234 slots).
- Inside bridged one-slot gaps, 16.9% of member-slots logged no scan, against 3.0% for a phone active in the slots before and after (about six times more). Still, 83% did log scans, so a gap is usually not a switched-off phone.

**Rejected:** θ ≤ 0.6 (drift), θ ≥ 0.8 (fragmentation), g = 0 (one missed scan splits a group), g ≥ 2 (little gain, more drift).

**Settled by the checks:** long episodes are overnight co-location, kept and flagged (section 7); one-slot episodes are kept (section 8); the effect of the RSSI threshold is in section 9.

**Limits:** hours assume that timestamp 0 is midnight; Bluetooth cannot distinguish a residence from the campus; the 3.0% baseline is exact only for one-slot gaps.

## 11. Schema of `episodes_v1`

One row per episode, built from `gatherings_v1` at the chosen setting (θ = 0.7, g = 1).

| Column | Meaning |
|---|---|
| `episode_id` | running id |
| `first_slot`, `last_slot` | first and last slot that has a row |
| `start_time`, `end_time` | `start_time` of the first row, `end_time` of the last row |
| `n_slots` | slots covered from first to last, gaps included (duration in 5-minute slots) |
| `n_gap_slots` | covered slots without a row (rows = `n_slots` − `n_gap_slots`) |
| `gathering_ids` | the `gathering_id` of every row in the chain |
| `participant_set` | everyone who appears in any row |
| `median_size` | median participants per row |
| `median_rssi` | median of the rows' `median_rssi` |
| `start_hour` | hour of the first slot (0–23); assumes timestamp 0 = midnight |
| `night_share` | share of covered slots between 00:00 and 07:00 |

Night-time rule: flag columns only, no filter in v1.